In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bokeh.plotting import figure, show
from bokeh.resources import INLINE
from bokeh.io import output_notebook
output_notebook(INLINE)
from os.path import join

Loading BokehJS ...

In [3]:
sample_list = list(pd.read_csv('/path/to/data/1KG_data/sample_list.txt', header=None).iloc[:,0])
sample = sample_list[0]
basedir = '/path/to/data/1KG_data/roh_intersect_EGA/'
chr = 9
f = join(basedir, f"chr{chr}", f"{sample}_roh.txt.gz")


In [4]:
# use comment to skip "RG" lines, this is obviously just a temoporary fix
df = pd.read_csv(f, sep='\t', skiprows=5, comment='R', header=None)
df.columns = ['class', 'sample', 'chr', 'pos', 'roh', 'prob']
df_sub = df.loc[(df.pos>2000000) & (df.pos<8000000)]

In [5]:
p = figure(plot_height = 300, plot_width = 600, title=sample, x_axis_label=f'Chr{chr} genomic_coordinate', y_axis_label='ROH Model')
p.line(df_sub.pos, df_sub.roh, line_width=2)
show(p)

In [6]:
p = figure(plot_height = 300, plot_width = 1000, title=sample, x_axis_label='Chr9 genomic_coordinate', y_axis_label='ROH Model')
p.line(df.pos, df.roh, line_width=2)
show(p)

In [23]:
# what's the total amount of LOH that is encompassed in this sample?
def calculate_roh_df(df):
    current_state=0
    counter=0
    roh_starts = []
    roh_ends = []
    for roh, pos in zip(df.roh, df.pos):
        if current_state==0:
            if roh ==1:
                counter +=1 
                roh_starts.append(pos)
                current_state=1
        else: 
            if roh==0:
                current_state=0
                roh_ends.append(pos)
    roh_calcs = pd.DataFrame({'start': roh_starts[0:len(roh_ends)], 
                          'end' : roh_ends[0:len(roh_ends)]})
    roh_calcs['len'] = [a-b for a,b in zip(roh_calcs['end'], roh_calcs['start'])]    
    return(roh_calcs)

In [26]:
basedir = '/path/to/data/1KG_data/roh_intersect_EGA/'
chr_list = list(range(1,23))
chr_list = [8,9,10]

# do this for each chromosome 
for chr in chr_list:
    print(chr)
    roh_calcs_dict = {}
    total_roh_length_dict = {}
    roh_number_dict = {}
    for i in range(len(sample_list)):
        sample = sample_list[i]
        print(f"  {i+1} out of {len(sample_list)}")
        f = join(basedir, f"chr{chr}", f"{sample}_roh.txt.gz")
        df = pd.read_csv(f, sep='\t', skiprows=5, comment='R', header=None)
        df.columns = ['class', 'sample', 'chr', 'pos', 'roh', 'prob']
        roh_calcs = calculate_roh_df(df)
        total_roh_length = np.sum(roh_calcs['len'])
        roh_number = roh_calcs.shape[0]    
        total_roh_length_dict[sample] = total_roh_length
        roh_number_dict[sample] = roh_number
    roh_stats_df = pd.DataFrame({'sample': sample_list,
                             'roh_number': [roh_number_dict[s] for s in sample_list],
                             'roh_total_length': [total_roh_length_dict[s] for s in sample_list]})
    roh_stats_df.index = roh_stats_df['sample']
    outf = join(basedir, 'roh_stats_df', f"chr{chr}_roh_stats.txt")
    roh_stats_df.to_csv(outf, sep='\t')

8
  1 out of 2504
  2 out of 2504
  3 out of 2504
  4 out of 2504
  5 out of 2504
  6 out of 2504
  7 out of 2504
  8 out of 2504
  9 out of 2504
  10 out of 2504
  11 out of 2504
  12 out of 2504
  13 out of 2504
  14 out of 2504
  15 out of 2504
  16 out of 2504
  17 out of 2504
  18 out of 2504
  19 out of 2504
  20 out of 2504
  21 out of 2504
  22 out of 2504
  23 out of 2504
  24 out of 2504
  25 out of 2504
  26 out of 2504
  27 out of 2504
  28 out of 2504
  29 out of 2504
  30 out of 2504
  31 out of 2504
  32 out of 2504
  33 out of 2504
  34 out of 2504
  35 out of 2504
  36 out of 2504
  37 out of 2504
  38 out of 2504
  39 out of 2504
  40 out of 2504
  41 out of 2504
  42 out of 2504
  43 out of 2504
  44 out of 2504
  45 out of 2504
  46 out of 2504
  47 out of 2504
  48 out of 2504
  49 out of 2504
  50 out of 2504
  51 out of 2504
  52 out of 2504
  53 out of 2504
  54 out of 2504
  55 out of 2504
  56 out of 2504
  57 out of 2504
  58 out of 2504
  59 out of 2504
  60

In [28]:
roh_stats_df

,sample,roh_number,roh_total_length
sample,,,
HG00096,HG00096,291,39455888
HG00097,HG00097,323,40264343
HG00099,HG00099,320,35382934
HG00100,HG00100,326,42414923
HG00101,HG00101,349,39915880
...,...,...,...
NA21137,NA21137,326,35139158
NA21141,NA21141,308,37757572
NA21142,NA21142,344,43510501
